# D4 · Land-cover classification: forest, fields, sediment and water

> **Time:** about 3 hours  
> **You need:** Parts A, B and C, and Lessons D1 to D3: bands with names and reflectance (D1), `normalized_difference`, NDVI and MNDWI (D2), the flood strip of D3. From Part A: array shapes (A8), and pandas tables with `groupby` (A9).  
> **By the end you can:**
> 1. turn a satellite image into a **table of features** (one row per cell) and back into a map, and group its cells with **k-means**,
> 2. train a **random forest** on a balanced sample of **reference labels**, predict every cell, and test it honestly with a **confusion matrix**, precision and recall, and a **spatial** split,
> 3. apply the model to another date, read a land-cover change table with care, and save the classified map as a GeoTIFF with its colours.

In D2 and D3 you chose the rules yourself: *NDVI below 0.3 means little vegetation*. **Classification** hands part of that work to the computer. Every cell gets a **class**, such as forest, cropland or water, decided from its **features**: its 10 band reflectances and two indices.

There are two ways to do it:

- **unsupervised**: the computer finds groups of similar cells on its own; *you* name the groups afterwards;
- **supervised**: you show the computer labelled examples ("this cell is cropland"), it learns from them, and labels every other cell.

This lesson uses **scikit-learn** (imported as `sklearn`), the standard Python library for machine learning. It is new to you; the lesson introduces each part as it comes.

The labelled examples come from **ESA WorldCover 2021**, a global land-cover map. WorldCover is itself the output of a model, made from a whole year of satellite images. It has errors and it is **not ground truth**. This lesson calls it **reference labels**: good enough to learn from and to compare with, never beyond question.

A common mistake in classification is to test a model on the same cells it learned from. This lesson shows why that misleads, and how to test fairly.

*Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.*  
*Reference labels: © ESA WorldCover project 2021 / Contains modified Copernicus Sentinel data (2021) processed by ESA WorldCover consortium (CC BY 4.0).*  
The first credit line belongs wherever the images, or maps made from them, are shown; the second wherever WorldCover, or a model trained on it, is shown.

**Counting lines:** lines that only set up or show a plot (`fig, ax = plt.subplots()`, `plt.show()`) do not count toward the Q1/Q2/Q3 line counts.

**How to practise:** after every step there are three questions. **Q1** needs one line of code, **Q2** two lines, **Q3** about three. Type your answer in the empty cell under each question, run it, then open **Show solution** to check. Try first, even if you are not sure: struggling for a few minutes is how the skill grows.

<!-- pygile-nav -->
**PyGILE Course** · lesson 36 of 36 · [Course home](../README.md)  
← Previous: [D3 · Change detection: the 2021 Melamchi flood](D03_Change_detection_Melamchi_flood.ipynb) · You have reached the end of the course

## 1. The image as a table of features

A classifier does not see a map. It sees a **table**: one **row per cell**, one **column per feature**. A **feature** is any number that describes a cell: here the 10 band reflectances, NDVI and MNDWI (D2). Twelve features in all.

The image is a 3-D array of shape **(bands, rows, columns)**, (10, 1006, 702). Two steps turn it into a table:

1. `cube.reshape(10, -1)` keeps the 10 bands and lays each band's 1,006 × 702 grid out as one long line of 706,212 values. `-1` means "work out this size yourself".
2. `.T` (**transpose**) swaps rows and columns: now each **row** is a cell and each **column** a band.

Cells are laid out **row by row**, like reading a page: cell number = row × 702 + column. So `values.reshape(1006, 702)` turns any column of the table back into a map. The tiny made-up example shows the idea first.

The first cell also repeats the helpers of D1 to D3 (`open_s2`, `get`, `composite`, `normalized_difference`), so this lesson stands on its own. Run it first.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
import rioxarray as rxr
from matplotlib.colors import ListedColormap, to_rgb
from matplotlib.patches import Patch
from scipy import ndimage

DATA = Path("..") / "data"
OUT = Path("..") / "outputs"
OUT.mkdir(exist_ok=True)
BEFORE = DATA / "s2_melamchi_before.tif"
AFTER = DATA / "s2_melamchi_after.tif"
S2_CREDIT = "Contains modified Copernicus Sentinel data 2020–2021, processed by ESA."
WC_CREDIT = ("© ESA WorldCover project 2021 / Contains modified Copernicus Sentinel data (2021) "
             "processed by ESA WorldCover consortium (CC BY 4.0).")


def open_s2(path):
    """Open a course Sentinel-2 file as reflectance (float32, NaN for nodata), with named bands (D1, D2)."""
    image = rxr.open_rasterio(path, masked=True)
    image = image.assign_coords(band=list(image.attrs["long_name"])) * 0.0001   # offset 0 for these dates
    image.attrs = {}
    return image


def get(image, name):
    """One band as a NumPy array (D3)."""
    return image.sel(band=name).values


def composite(image, band_names, tops=None):
    """Three bands as an RGB picture, each divided by its top value and clipped to 0-1 (D1)."""
    rgb = np.dstack([get(image, name) for name in band_names])
    if tops is None:
        tops = np.nanpercentile(rgb, 98, axis=(0, 1))
    return np.clip(np.nan_to_num(rgb / tops), 0, 1)


def normalized_difference(a, b):
    """(a - b) / (a + b) for each cell, with NaN where a + b is 0 (D2)."""
    total = a + b
    total = np.where(total == 0, np.nan, total)     # guard: no division by zero
    return (a - b) / total

In [ ]:
# A tiny made-up image: 2 bands, 2 rows, 3 columns
tiny = np.array([[[1, 2, 3],
                  [4, 5, 6]],          # band 1
                 [[10, 20, 30],
                  [40, 50, 60]]])      # band 2
print("image shape (bands, rows, cols):", tiny.shape)

tiny_table = tiny.reshape(2, -1).T     # one row per cell, one column per band
print(pd.DataFrame(tiny_table, columns=["band 1", "band 2"]))

print("band 2 back as a map:")
print(tiny_table[:, 1].reshape(2, 3))

In [ ]:
before = open_s2(BEFORE)
after = open_s2(AFTER)
n_rows, n_cols = after.shape[1], after.shape[2]
BANDS = [str(name) for name in after.band.values]      # plain Python strings for column names
FEATURES = BANDS + ["NDVI", "MNDWI"]


def feature_table(image):
    """One row per cell, one column per feature: 10 bands, NDVI and MNDWI."""
    table = pd.DataFrame(image.values.reshape(len(BANDS), -1).T, columns=BANDS)
    table["NDVI"] = normalized_difference(table["B08"].values, table["B04"].values)
    table["MNDWI"] = normalized_difference(table["B03"].values, table["B11"].values)
    return table


after_table = feature_table(after)
before_table = feature_table(before)
valid = after_table.notna().all(axis=1).to_numpy()     # True where all 12 features have a value
print(after_table.shape, "| cells with all features:", valid.sum(), "| with a gap:", (~valid).sum())
after_table.head(3).round(3)

The tiny image has the shape (2, 2, 3). Its table has **6 rows**, one per cell, and **2 columns**, one per band. Row 0 is the top-left cell (1 and 10), row 3 the first cell of the second row (4 and 40): row by row. Column 1 reshaped to (2, 3) gives band 2 back as a map, unchanged.

The real table has **706,212 rows** (1,006 × 702 cells) and **12 columns**. 706,067 cells have all 12 features. The other 145 are the shadow cells of the after image with a missing band (D1); a classifier cannot use a row with a gap, so the lesson keeps a True/False list, `valid`, and leaves those cells out.

`BANDS` turns the band names into plain Python strings with `str()`. The names stored in the DataArray are NumPy strings, and scikit-learn refuses a table whose column names are a mix of the two kinds.

`.notna().all(axis=1)` is `True` for a row only if **all** its values are present: `axis=1` looks across the columns of each row (A9).

### Practice

**Q1 · one line:** Which features have missing values in `after_table`, and how many? Show the result of `.isna().sum()` for the whole table.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
after_table.isna().sum()
```

B02 has 12, B03 25, B04 125 and B08 3 missing values; NDVI has 128 (it needs B04 and B08) and MNDWI 25 (it needs B03). The 145 invalid rows are the cells where at least one of these is missing.
</details>

**Q2 · two lines:** Cell number 300,000: which row and column of the map is it? Use `//` and `%` (A2) with `n_cols`, and check by comparing its NDVI in the table with the NDVI grid of D2 at that row and column.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
row, col = 300000 // n_cols, 300000 % n_cols
print(row, col, after_table.loc[300000, "NDVI"], normalized_difference(get(after, "B08"), get(after, "B04"))[row, col])
```

Row 427, column 246, and both NDVIs agree. Cell number = row × 702 + column, so `//` gives the row and `%` the column that is left over.
</details>

**Q3 · three lines:** Turn the whole `NDVI` column back into a map with `.reshape(n_rows, n_cols)`, and check with `np.allclose(..., equal_nan=True)` (D2) that it equals the NDVI grid computed directly from the bands.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
ndvi_map = after_table["NDVI"].to_numpy().reshape(n_rows, n_cols)
ndvi_grid = normalized_difference(get(after, "B08"), get(after, "B04"))
print(ndvi_map.shape, np.allclose(ndvi_map, ndvi_grid, equal_nan=True))
```

(1006, 702) and `True`: from map to table and back loses nothing, as long as you reshape with the same numbers of rows and columns.
</details>

## 2. Reference labels: ESA WorldCover 2021

Supervised classification needs **labels**: cells whose class you already know. Here they come from **ESA WorldCover 2021**, saved on exactly the image grid by `../data/get_worldcover_melamchi.py`. Each cell holds a class **code**:

| code | class | official colour |
| --- | --- | --- |
| 10 | tree cover | dark green |
| 30 | grassland | yellow |
| 40 | cropland | pink |
| 50 | built-up | red |
| 60 | bare / sparse vegetation | grey |
| 80 | permanent water | blue |

Codes are **names**, not amounts: 50 is not "more" than 40. So a class map needs one fixed colour per code, not a colour scale. The example builds a **lookup table**: an array with one row per possible code (0 to 255), each holding a colour. `lut[grid]` replaces every code in the grid by its row of the table, which gives a colour picture (rows, columns, 3), as `np.dstack` did in D1.

WorldCover was made from a whole year of 2021 images, by a model, at 10 m (here reduced to the most common class in each 20 m cell). Treat it as **reference labels**, not truth: you will meet its errors.

In [ ]:
with rasterio.open(DATA / "worldcover_melamchi_2021.tif") as src:
    worldcover = src.read(1)
    transform = src.transform
    profile = src.profile
    print(src.crs, src.res, worldcover.shape, worldcover.dtype, "| nodata:", src.nodata)
    print("same grid as the image:", src.transform == after.rio.transform(), src.shape == (n_rows, n_cols))

NAMES = {10: "tree cover", 30: "grassland", 40: "cropland", 50: "built-up", 60: "bare / sparse", 80: "water"}
COLOURS = {10: "#006400", 30: "#ffff4c", 40: "#f096ff", 50: "#fa0000", 60: "#b4b4b4", 80: "#0064c8"}
CODES = list(NAMES)
CELL_KM2 = 20 * 20 / 1000000            # one 20 m cell is 0.0004 km² (D2)

labels = worldcover.ravel()              # one label per row of the feature tables
areas = pd.DataFrame({"class": [NAMES[c] for c in CODES],
                      "cells": [(labels == c).sum() for c in CODES]}, index=CODES)
areas["area_km2"] = areas["cells"] * CELL_KM2
areas["percent"] = areas["cells"] / labels.size * 100
areas.round(2)

In [ ]:
lut = np.ones((256, 3))                  # white for every code ...
for code, colour in COLOURS.items():
    lut[code] = to_rgb(colour)           # ... except the six classes
legend = [Patch(color=COLOURS[c], label=f"{c} {NAMES[c]}") for c in CODES]


def legend_and_credits(fig):
    """Class legend and both credit lines along the bottom of a figure of class maps."""
    fig.subplots_adjust(bottom=0.15)
    fig.legend(handles=legend, loc="lower center", bbox_to_anchor=(0.5, 0.045), ncol=6, fontsize=9)
    fig.text(0.01, 0.005, S2_CREDIT + "\n" + WC_CREDIT, fontsize=7)


left, bottom, right, top = after.rio.bounds()
extent_km = [left / 1000, right / 1000, bottom / 1000, top / 1000]
TRUE_COLOUR = ["B04", "B03", "B02"]
tops = np.nanpercentile(np.dstack([get(before, b) for b in TRUE_COLOUR]), 98, axis=(0, 1))   # D1's shared stretch

fig, axes = plt.subplots(1, 2, figsize=(10.5, 8.5))   # fig: the whole figure; axes: its two map panels (A10)
axes[0].imshow(composite(after, TRUE_COLOUR, tops), extent=extent_km)
axes[0].set_title("Sentinel-2 true colour, 11 November 2021")
axes[1].imshow(lut[worldcover], extent=extent_km, interpolation="nearest")
axes[1].set_title("ESA WorldCover 2021 (reference labels)")
for ax in axes:
    ax.set_xlabel("Easting (km)")
axes[0].set_ylabel("Northing (km)")
legend_and_credits(fig)
plt.show()

In [ ]:
# Where does WorldCover put D3's flood strip? (D3's recipe, in short)
d_ndvi = (after_table["NDVI"] - before_table["NDVI"]).to_numpy().reshape(n_rows, n_cols)
change = d_ndvi < -0.3                                    # D3's threshold
EIGHT = np.ones((3, 3), dtype=bool)
groups, n_groups = ndimage.label(ndimage.binary_dilation(change, structure=EIGHT), structure=EIGHT)
groups = np.where(change, groups, 0)
ids, counts = np.unique(groups[groups > 0], return_counts=True)
strip = groups == ids[counts.argmax()]                    # the largest group: the main strip
print("strip:", strip.sum(), "cells =", round(strip.sum() * CELL_KM2, 2), "km²")

in_strip = pd.Series(worldcover[strip]).map(NAMES).value_counts()
print((in_strip / strip.sum() * 100).round(1))

The file is on exactly the image grid: same transform, same 1,006 × 702 shape, `uint8` codes, nodata 0. Every row of the feature tables now has a label: `labels` is the map laid out row by row, like the tables.

The area table shows a strongly **unbalanced** landscape: tree cover holds 551,251 cells, about **220.5 km²** or **78.1%** of the box. Grassland follows with about 42.8 km², cropland 11.2 km², bare or sparse vegetation 4.9 km², built-up 2.3 km² and permanent water only about **0.74 km²** (1,862 cells, 0.3%). Step 4 has to deal with this.

The two maps agree in their broad lines: dark forest on the slopes, patches of grassland scattered over them, cropland mostly near the rivers, a grey ribbon of bare ground along the Melamchi river and a red built-up patch a little south of where the two rivers meet. `legend_and_credits(fig)` adds the class legend and both credit lines under any figure of class maps; the lesson uses it for every such figure.

The **main strip** of D3, its largest group of changed cells, holds 7,727 cells (3.09 km²); this lesson calls it the flood strip, `strip`. D3's final estimate, about 3.2 km², adds three small groups on the valley floor. WorldCover calls only about **51.6%** of it bare or sparse vegetation, and about 5.9% water. The rest is grassland (14.0%), cropland (13.0%), tree cover (11.8%) and built-up (3.7%), although the November 2021 image shows bare sediment over most of it. Two likely reasons, which this data cannot prove: WorldCover 2021 was made from images of the whole year, and the flood came in June, so part of the year still showed the old fields and trees; and edge cells of a strip only about 7 cells wide (D3) mix sediment with vegetation. Either way: **the reference labels are not right everywhere.** A model that learns from them learns their mistakes too.

*Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.*  
*Reference labels: © ESA WorldCover project 2021 / Contains modified Copernicus Sentinel data (2021) processed by ESA WorldCover consortium (CC BY 4.0).*

### Practice

**Q1 · one line:** What percentage of the cells does WorldCover call tree cover? Use `labels == 10` and `.mean()`, rounded to 1 decimal.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
round((labels == 10).mean() * 100, 1)
```

About 78.1%. Remember this number: a map that says "tree cover" everywhere, without looking at the image at all, would agree with WorldCover in 78.1% of the cells.
</details>

**Q2 · two lines:** What share of all WorldCover **bare / sparse** cells lies inside the flood strip, and what share of all **water** cells? Print both percentages, rounded to 1 decimal.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(round((strip & (worldcover == 60)).sum() / (worldcover == 60).sum() * 100, 1))
print(round((strip & (worldcover == 80)).sum() / (worldcover == 80).sum() * 100, 1))
```

About 32.3% of the bare / sparse cells and 24.4% of the water cells lie in the strip. The strip is a large part of these rare classes, but most bare ground lies elsewhere: dry fields, landslide scars, river banks and rock.
</details>

**Q3 · three lines:** Zoom in on the place where the rivers meet: draw the after image and WorldCover side by side with `ax.axis([357.5, 361, 3077, 3081.5])` (km), and finish with `legend_and_credits(fig)`.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, axes = plt.subplots(1, 2, figsize=(10, 7.5))
axes[0].imshow(composite(after, TRUE_COLOUR, tops), extent=extent_km)
axes[1].imshow(lut[worldcover], extent=extent_km, interpolation="nearest")
for ax in axes:
    ax.axis([357.5, 361, 3077, 3081.5])
axes[0].set_title("11 November 2021")
axes[1].set_title("WorldCover 2021")
legend_and_credits(fig)
plt.show()
```

Close up, WorldCover's grey and blue follow the sediment north of the confluence. South of it, much of the white sediment is labelled grassland (yellow) or cropland (pink), and a red built-up patch sits a little south of the place where the rivers meet. Comparing a label map with the image it should describe is the quickest way to judge where the labels can be trusted.
</details>

## 3. Unsupervised: k-means finds groups

**scikit-learn** works the same way for every method. A **model** is a recipe with numbers it can learn from data. Every model has two main methods:

- `model.fit(X)` **learns** from the table `X` (rows = cells, columns = features).
- `model.predict(X)` **applies** what it learned to any table with the same columns, and returns one answer per row.

**k-means** is an unsupervised model. You choose the number of groups, **k**. It places k centres among the cells, gives every cell to its nearest centre, moves each centre to the mean of its cells, and repeats until nothing moves. The groups are called **clusters**.

"Nearest" is a distance between features, so the features need comparable sizes. Reflectances lie between about 0 and 0.5, the indices between −1 and 1. `StandardScaler` rescales each feature to a mean of 0 and a standard deviation of 1. A **pipeline** chains the steps, here *scale, then cluster*, into one model, so they always run together and in the same order, on the sample and on every later table.

`n_init=10` runs k-means from 10 starting positions and keeps the best; `random_state=0` fixes the random starts, so you get the same clusters every time. Fitting on a random sample of 20,000 cells is fast; predicting all cells is then quick.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

sample = after_table[valid].sample(20000, random_state=0)      # 20,000 random cells with all features

kmeans_model = Pipeline([
    ("scale", StandardScaler()),                                 # step 1: every feature to mean 0, std 1
    ("kmeans", KMeans(n_clusters=8, n_init=10, random_state=0)),  # step 2: 8 clusters
])
kmeans_model.fit(sample)                                         # learn from the sample

clusters = np.full(len(after_table), np.nan)                     # NaN for the 145 invalid cells
clusters[valid] = kmeans_model.predict(after_table[valid])       # one cluster number per cell
cluster_map = clusters.reshape(n_rows, n_cols)
print(pd.Series(clusters[valid]).value_counts().sort_index())

fig, ax = plt.subplots(figsize=(7, 9))
im = ax.imshow(cluster_map, extent=extent_km, cmap=ListedColormap(plt.cm.tab10.colors[:8]),
               vmin=-0.5, vmax=7.5, interpolation="nearest")
fig.colorbar(im, ax=ax, ticks=range(8), shrink=0.6, label="Cluster number")
ax.set_title("k-means, 8 clusters, 11 November 2021")
ax.set_xlabel("Easting (km)")
ax.set_ylabel("Northing (km)")
fig.text(0.01, 0.02, S2_CREDIT, fontsize=8)
plt.show()

In [ ]:
# Clusters are not classes: name them by comparing with the reference labels
table = pd.crosstab(clusters[valid].astype(int), pd.Series(labels[valid]).map(NAMES),
                    rownames=["cluster"], colnames=["WorldCover"], normalize="index") * 100
table = table.round(1)
table["mean NIR"] = after_table[valid].groupby(clusters[valid].astype(int))["B08"].mean().round(3)
table

The eight clusters range from 6,431 to 261,475 cells. On the map they form sensible patterns: the valley floor, the fields, the sunlit and the shaded slopes each take their own colours.

But a cluster number is **not a class**. The cross-tabulation (`pd.crosstab`, like a pivot table of counts, A9) shows, for each cluster, which WorldCover classes its cells belong to; `normalize="index"` turns each row into percentages.

- **Four clusters** (1, 2, 3 and 6) are 81% to 99% tree cover. They differ above all in brightness: their mean NIR runs from about 0.15 to about 0.45. k-means split the forest by **illumination**, shaded slopes from sunlit ones (D3's false change, again).
- Two clusters (0 and 5) are mixtures of grassland, tree cover and cropland, with no class above 50%.
- Two clusters (4 and 7) have bare / sparse as their largest class, but neither is pure: cluster 4 is about 47% bare / sparse, 16% built-up and 12% water (the river bed and the settlements beside it); cluster 7 is about 56% bare / sparse, with grassland and cropland.

So k-means found the groups the **image** contains, not the classes **you** care about. You, the analyst, must name them, and several clusters may share one name ("tree cover") while one cluster mixes several classes. That is the honest limit of unsupervised classification.

*Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.*  
*Reference labels: © ESA WorldCover project 2021 / Contains modified Copernicus Sentinel data (2021) processed by ESA WorldCover consortium (CC BY 4.0).*

### Practice

**Q1 · one line:** Which clusters fill D3's flood strip? Count the clusters of the strip cells with `pd.Series(cluster_map[strip]).value_counts()`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
pd.Series(cluster_map[strip]).value_counts()
```

Cluster 7 holds 3,764 strip cells and cluster 4 holds 3,063: together 6,827 of the 7,727 cells. k-means found the sediment strip without being told it exists. But neither cluster is the strip: each also holds bare ground, buildings and water elsewhere.
</details>

**Q2 · two lines:** A pipeline keeps its steps by name. Get the cluster centres back in reflectance units: `kmeans_model["scale"].inverse_transform(kmeans_model["kmeans"].cluster_centers_)`, put them in a DataFrame with `columns=FEATURES`, and show the `B08`, `B11`, `NDVI` and `MNDWI` columns rounded to 2 decimals.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
centres = pd.DataFrame(kmeans_model["scale"].inverse_transform(kmeans_model["kmeans"].cluster_centers_), columns=FEATURES)
centres[["B08", "B11", "NDVI", "MNDWI"]].round(2)
```

The centres are the mean features of each cluster. The two bare clusters, 4 and 7, have by far the lowest NDVI (about 0.11 and 0.06) and the highest MNDWI (about −0.21 and −0.18). The four forest clusters all have an NDVI of about 0.80 to 0.82, but NIR from about 0.15 to 0.45. `inverse_transform` undoes the scaling, so the numbers are reflectances and index values again.
</details>

**Q3 · three lines:** Try **k = 4**: build a new pipeline with 4 clusters (same `n_init` and `random_state`), fit it on `sample`, predict `sample`, and cross-tabulate with the sample's WorldCover labels (`labels[sample.index]`), as row percentages.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
model_4 = Pipeline([("scale", StandardScaler()), ("kmeans", KMeans(n_clusters=4, n_init=10, random_state=0))])
model_4.fit(sample)
print((pd.crosstab(model_4.predict(sample), pd.Series(labels[sample.index]).map(NAMES).values, normalize="index") * 100).round(0))
```

With only 4 clusters, two are still mostly tree cover (about 94% and 78%), one mixes grassland, tree cover and cropland, and one is about 46% bare / sparse. Built-up and water never get a cluster of their own. Fewer clusters do not mean cleaner classes. `sample.index` holds the cell numbers of the sampled rows, so `labels[sample.index]` gives their labels.
</details>

## 4. Supervised: a random forest learns from labelled cells

A supervised model learns from examples with answers. By custom, `X` is the table of features and `y` the list of labels, and `model.fit(X, y)` learns how features lead to labels.

A **decision tree** is a chain of yes/no questions, such as *"NDVI above 0.62? If yes: B11 below 0.21? …"*, learned so that each answer separates the classes as well as possible. One tree tends to memorise its examples. A **random forest** grows many trees, each from a random part of the examples and of the features, and lets them **vote**: the class with most votes wins. `n_estimators=100` grows 100 trees; `random_state=0` makes the result repeatable; `n_jobs=-1` uses all processor cores.

**Which cells to learn from?** A random sample of the image would be about 78% tree cover and hold only a handful of water cells. The forest would learn trees very well and water hardly at all. A **stratified** sample takes the same number from each class: here **1,000 cells per class**, 6,000 in all. `groupby(...).sample(n=1000, random_state=0)` does it in one line (each class here has more than 1,000 cells).

Trees compare one feature with a threshold at a time, so they need no scaling.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

cells = after_table[valid].copy()
cells["worldcover"] = labels[valid]
print("a random sample of 6,000 cells would hold:", cells.sample(6000, random_state=0)["worldcover"].value_counts().to_dict())

train = cells.groupby("worldcover").sample(n=1000, random_state=0)   # 1,000 cells of each class
X = train[FEATURES]
y = train["worldcover"]
print("the stratified sample holds:", y.value_counts().to_dict())

forest = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1)
forest.fit(X, y)                                                      # learn from 6,000 labelled cells

predicted = np.zeros(len(after_table), dtype="uint8")                 # 0 = nodata
predicted[valid] = forest.predict(after_table.loc[valid, FEATURES])   # all 706,067 valid cells
rf_map = predicted.reshape(n_rows, n_cols)
print("classes in the map:", np.unique(rf_map))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 8.5))
axes[0].imshow(lut[rf_map], extent=extent_km, interpolation="nearest")
axes[0].set_title("Random forest, 11 November 2021")
axes[1].imshow(lut[worldcover], extent=extent_km, interpolation="nearest")
axes[1].set_title("ESA WorldCover 2021 (reference labels)")
for ax in axes:
    ax.set_xlabel("Easting (km)")
axes[0].set_ylabel("Northing (km)")
legend_and_credits(fig)
plt.show()

A plain random sample of 6,000 cells would hold 4,703 tree cells and only 21 water cells. The stratified sample holds exactly 1,000 of each class.

The forest predicts all 706,067 valid cells in a few seconds; `lut[rf_map]` paints them, and code 0 (nodata) stays white.

The two maps tell the same broad story: forest on the slopes, grassland patches, bare ground and water along the river, built-up where the rivers meet. But they differ in two ways you can see. Along the Melamchi river the forest paints a wide **blue** ribbon of water where WorldCover has grey bare ground. And on the slopes it has **far more pink (cropland) and red (built-up)** cells, scattered as specks. It learned from a sample in which each rare class was as common as tree cover, so it is quick to call a cell "built-up" or "water". A balanced sample helps the forest to *find* rare classes, at the price of *false alarms*. Step 5 measures both.

`after_table.loc[valid, FEATURES]` selects the valid rows and the feature columns in the order the forest learned them.

*Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.*  
*Reference labels: © ESA WorldCover project 2021 / Contains modified Copernicus Sentinel data (2021) processed by ESA WorldCover consortium (CC BY 4.0).*

### Practice

**Q1 · one line:** How many cells of each class did the forest predict? Use `pd.Series(predicted[valid]).value_counts()`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
pd.Series(predicted[valid]).value_counts()
```

Tree cover 457,189 cells, grassland 148,654, cropland 58,788, built-up 21,797, bare / sparse 12,475 and water 7,164. Compare with WorldCover's 5,685 built-up and 1,862 water cells: the forest calls nearly four times as many cells built-up or water.
</details>

**Q2 · two lines:** What does the forest say at D1's four cells? For each `(x, y)` in `[(361010, 3089990), (359610, 3079190), (359710, 3076730), (359610, 3077990)]` (forest, built-up area, river, flood sediment), find the row and column with `rasterio.transform.rowcol(transform, x, y)` (C1), and print the forest's class name and WorldCover's.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
for x, y_m in [(361010, 3089990), (359610, 3079190), (359710, 3076730), (359610, 3077990)]:
    row, col = rasterio.transform.rowcol(transform, x, y_m)
    print(NAMES[rf_map[row, col]], "| WorldCover:", NAMES[worldcover[row, col]])
```

Tree cover, bare / sparse, water and water. The forest and the river cells get the classes their profiles suggested. The built-up cell (WorldCover: built-up) is called bare / sparse: its flat, bright profile in D1 mixed roofs with bare ground. The flood-sediment cell is called water, perhaps because the sediment was wet; WorldCover itself calls that cell grassland. Neither map is right everywhere. The loop variable is called `y_m`, not `y`, so that it does not overwrite the labels `y`.
</details>

**Q3 · three lines:** Fit a second forest with only **10 trees** (`n_estimators=10`, same `random_state`), predict the valid cells, and print the percentage of valid cells on which it agrees with the 100-tree forest, rounded to 1 decimal.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
small_forest = RandomForestClassifier(n_estimators=10, random_state=0, n_jobs=-1).fit(X, y)
small_pred = small_forest.predict(after_table.loc[valid, FEATURES])
print(round((small_pred == predicted[valid]).mean() * 100, 1))
```

About 92.4% agreement: most cells get the same class from 10 or 100 trees, but 7.6% of the cells do not. Votes of more trees are more stable; 100 is a common, modest choice.
</details>

## 5. How good is it? Accuracy, confusion matrix, precision and recall

Never judge a model on the cells it learned from: it has seen their answers. The fair test is a **hold-out**: keep some labelled cells apart, train on the rest, and compare predictions with the labels of the cells kept apart. `train_test_split` splits at random; `stratify=y` keeps 1,000 : 1,000 proportions in both parts.

Three ways to read the result:

- **Overall accuracy**: the share of test cells whose predicted class equals the reference label.
- The **confusion matrix**: a table of reference class (rows) against predicted class (columns). The diagonal holds the agreements; every other number is a particular mix-up.
- Per class: **recall** answers *"of the cells that ARE water, what share did the model find?"*, and **precision** answers *"of the cells the model CALLS water, what share are water?"*. `classification_report` prints both.

One warning before you run it. When one class dominates, overall accuracy flatters. A "map" that says *tree cover* everywhere agrees with WorldCover in 78.1% of the image (step 2, Q1), yet it finds no fields, no buildings and no water at all.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, recall_score, ConfusionMatrixDisplay

print("accuracy on its own training cells:", forest.score(X, y))      # a common mistake

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
test_forest = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1)
test_forest.fit(X_train, y_train)                                      # 4,200 cells
y_pred = test_forest.predict(X_test)                                   # 1,800 cells it has never seen
print("hold-out accuracy:", round(accuracy_score(y_test, y_pred), 3))

class_names = [NAMES[c] for c in CODES]
print(classification_report(y_test, y_pred, labels=CODES, target_names=class_names, digits=2))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, labels=CODES, display_labels=class_names,
                                        cmap="Blues", colorbar=False, ax=ax, xticks_rotation=45)
ax.set_title("Hold-out test: 300 cells per class")
ax.set_xlabel("Predicted by the random forest")
ax.set_ylabel("WorldCover reference label")
plt.show()

In [ ]:
# The whole map against the whole reference, leaving out the 6,000 training cells
untouched = valid.copy()
untouched[train.index] = False                       # train.index holds the cell numbers of the sample
agree = (predicted[untouched] == labels[untouched]).mean()
all_trees = (labels[untouched] == 10).mean()
print(f"forest map agrees with WorldCover:  {agree:.3f}")
print(f"'tree cover everywhere' agrees:     {all_trees:.3f}")
print(classification_report(labels[untouched], predicted[untouched], labels=CODES, target_names=class_names, digits=2))

On its own training cells the forest scores about **0.9998**: all but one of the 6,000 cells right. That is memory, not skill, and it says nothing about new cells. Testing on the training cells is a common mistake.

On the **1,800 hold-out cells** (300 per class) the accuracy is about **0.692**: about two cells in three get WorldCover's class. The report shows where it works: tree cover has a precision of 0.91 and a recall of 0.80, cropland a recall of 0.79 and water 0.77. **Bare / sparse** is the weakest class: recall only 0.40, precision 0.50.

The confusion matrix shows why. Read along a row: of the 300 bare / sparse test cells, 120 were predicted bare / sparse, and most of the others water or built-up. Grassland and cropland are confused with each other, and with tree cover. These classes are similar in the image, and the reference labels themselves are uncertain there (step 2 found grassland and cropland labels on flood sediment).

The **whole map** tells a second story. Among the cells it did not train on, it agrees with WorldCover in about **0.781** of them. The map that says "tree cover" everywhere agrees in about **0.786**: slightly *more*. (That is a little above step 2's 78.1%, because the 6,000 training cells left out are mostly not tree cover.) Yet the forest finds rare classes: its recall is about 0.67 to 0.82 for every class except bare / sparse (0.37). The price is **precision**: of the cells it calls water, only about **12%** are water in WorldCover, and only about 16% of the cells it calls built-up. Rare classes in a big landscape collect many false alarms.

So **one accuracy number is never enough**. Report the confusion matrix and the per-class precision and recall, and say which cells you tested on. `support` in the report is the number of reference cells of each class.

### Practice

**Q1 · one line:** What does the "tree cover everywhere" map score on the **balanced** hold-out cells? Compute the share of `y_test` that is 10.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
round((y_test == 10).mean(), 3)
```

About 0.167, one class in six. On a balanced test the lazy map is exposed at once; on the whole, unbalanced image it scores about 0.78. The same map, two very different scores: always say what the test cells are.
</details>

**Q2 · two lines:** Make the confusion matrix as a pandas table instead, with `pd.crosstab` of the class **names** of `y_test` (rows) and `y_pred` (columns), with `normalize="index"`, as percentages rounded to 0 decimals.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
names_test, names_pred = y_test.map(NAMES).values, pd.Series(y_pred).map(NAMES).values
print((pd.crosstab(names_test, names_pred, rownames=["reference"], colnames=["predicted"], normalize="index") * 100).round(0))
```

Each row adds up to 100% (give or take rounding), and the diagonal is the recall of each class: about 80% for tree cover, 40% for bare / sparse. The bare / sparse row spreads over water (about 27%), built-up (19%) and cropland (8%); 19% of the water cells were called bare / sparse. Wet gravel, dry gravel and shallow water grade into each other in 20 m cells.
</details>

**Q3 · three lines:** Compute the precision and recall of **built-up** on the whole map yourself: among `untouched` cells, precision = cells predicted 50 that are 50 in WorldCover ÷ cells predicted 50; recall = the same count ÷ cells that are 50 in WorldCover. Round to 2 decimals.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
both_50 = ((predicted == 50) & (labels == 50) & untouched).sum()
print(round(both_50 / ((predicted == 50) & untouched).sum(), 2))
print(round(both_50 / ((labels == 50) & untouched).sum(), 2))
```

Precision about 0.16 and recall about 0.70, as in the report. The forest finds about 7 in 10 built-up cells, but 5 of every 6 cells it calls built-up are something else in WorldCover: bright bare fields and gravel look like roofs in 20 m cells.
</details>

## 6. Neighbours are alike: a spatial test

A random hold-out has a hidden weakness on maps. Neighbouring cells are often much alike: same field, same forest, same light. If a test cell has a training cell right next to it, the model has partly seen the answer. This is called **leakage**, and it makes the score too optimistic.

The fix is a **spatial split**: cut the map into **blocks** and keep whole blocks apart, so a test cell has no training neighbour in its own block. Here the blocks are **2 km × 2 km** (100 × 100 cells): block number = (row // 100) × 10 + column // 100.

**Cross-validation** repeats a split several times so that every cell is tested once. With 5 **folds**, the cells are divided into 5 parts; the model trains on 4 parts and is tested on the fifth, 5 times. `cross_val_score` returns the 5 scores.

- `KFold(5, shuffle=True)` forms the folds from **random** cells.
- `GroupKFold(5)` with `groups=blocks` keeps every **block** whole inside one fold.

The same idea matters for every map model you build later, for example a landslide model, which must be tested on places it has not seen.

In [ ]:
ndvi_map = after_table["NDVI"].to_numpy().reshape(n_rows, n_cols)
pairs = pd.DataFrame({"cell": ndvi_map[:, :-1].ravel(), "east_neighbour": ndvi_map[:, 1:].ravel()}).dropna()
print("NDVI of a cell vs its east neighbour, correlation:", round(pairs["cell"].corr(pairs["east_neighbour"]), 3))

cell_row = train.index.to_numpy() // n_cols                 # cell number -> row and column (step 1)
cell_col = train.index.to_numpy() % n_cols
blocks = (cell_row // 100) * 10 + cell_col // 100           # 2 km blocks
print("training cells fall in", len(np.unique(blocks)), "blocks")

In [ ]:
from sklearn.model_selection import KFold, GroupKFold, cross_val_score, cross_val_predict

model = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1)
random_scores = cross_val_score(model, X, y, cv=KFold(5, shuffle=True, random_state=0))
spatial_scores = cross_val_score(model, X, y, cv=GroupKFold(5), groups=blocks)
print("random folds :", random_scores.round(3), "mean", round(random_scores.mean(), 3))
print("spatial folds:", spatial_scores.round(3), "mean", round(spatial_scores.mean(), 3))

In [ ]:
fold_of_block = {}
for fold, (train_rows, test_rows) in enumerate(GroupKFold(5).split(X, y, groups=blocks)):
    for b in np.unique(blocks[test_rows]):
        fold_of_block[b] = fold

fold_map = np.full((n_rows, n_cols), np.nan)
for b, fold in fold_of_block.items():
    r, c = b // 10, b % 10
    fold_map[r * 100:(r + 1) * 100, c * 100:(c + 1) * 100] = fold

fig, ax = plt.subplots(figsize=(7, 9))
ax.imshow(composite(after, TRUE_COLOUR, tops), extent=extent_km)
im = ax.imshow(fold_map, extent=extent_km, cmap=ListedColormap(plt.cm.Set2.colors[:5]), vmin=-0.5, vmax=4.5, alpha=0.45)
fig.colorbar(im, ax=ax, ticks=range(5), shrink=0.6, label="Test fold of each 2 km block")
ax.plot(cell_col * 0.02 + extent_km[0], extent_km[3] - cell_row * 0.02, ",", color="black")
ax.set_title("Spatial cross-validation: whole 2 km blocks are tested together")
ax.set_xlabel("Easting (km)")
ax.set_ylabel("Northing (km)")
fig.text(0.01, 0.02, S2_CREDIT, fontsize=8)
plt.show()

Neighbours are much alike: the NDVI of a cell and of its eastern neighbour have a correlation of about **0.80**. The 6,000 training cells fall in 82 of the 2 km blocks.

With **random** folds the mean accuracy is about **0.678**; with **spatial** folds it is about **0.650**. The spatial test is lower, as expected, but only by about 3 percentage points. Why so little? Our sample is thin: 6,000 cells spread over 706,212, so for most classes a test cell rarely has a training cell next to it, even in a random split. The leak should be strongest for the classes whose cells crowd together, such as **water**: 1,000 of only 1,862 water cells, all packed along a thin river. Q3 checks this.

On the map, each colour is one fold: all the black training dots in a coloured block are tested together, while the model learns from the other four colours. A few blocks hold no training cell and take no part.

Which score to report? The spatial one. It is closer to what a user of the map meets: a model applied to places it did not learn from.

*Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.*

### Practice

**Q1 · one line:** Print the standard deviation of the 5 spatial scores, rounded to 3 decimals. How much do the folds differ?

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
round(spatial_scores.std(), 3)
```

About 0.029: the five spatial folds score between about 0.61 and 0.70. The gap of about 3 points between the two tests is no larger than this spread between folds. Treat it as a hint that agrees with the leakage idea, not as an exact number.
</details>

**Q2 · two lines:** Does the block size matter? Make **1 km** blocks (50 × 50 cells: `(cell_row // 50) * 20 + cell_col // 50`) and print the mean spatial cross-validation accuracy with them, rounded to 3 decimals.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
blocks_1km = (cell_row // 50) * 20 + cell_col // 50
print(round(cross_val_score(model, X, y, cv=GroupKFold(5), groups=blocks_1km).mean(), 3))
```

About 0.663, between the 2 km blocks (0.650) and random folds (0.678). Smaller blocks keep fewer neighbours apart, so they leak more. The block size is a choice to state, like a threshold: blocks should be at least as large as the distance over which cells stay alike.
</details>

**Q3 · three lines:** `cross_val_predict` (imported in the example) gives one prediction per training cell, each from the fold in which that cell was tested. Use it with the random folds and with the spatial folds, and print the recall of each class with `recall_score(y, ..., labels=CODES, average=None)` (imported in step 5), rounded to 2 decimals. Give `groups` only to `GroupKFold`.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
for name, cv, g in [("random ", KFold(5, shuffle=True, random_state=0), None), ("spatial", GroupKFold(5), blocks)]:
    pred_cv = cross_val_predict(model, X, y, cv=cv, groups=g)
    print(name, recall_score(y, pred_cv, labels=CODES, average=None).round(2))
```

Tree cover and grassland keep the same recall (0.81 and 0.66), cropland drops from 0.72 to 0.70 and bare / sparse from 0.38 to 0.35, but built-up falls from 0.71 to 0.65 and **water** from 0.80 to 0.73. The classes whose cells crowd together, in villages and along a thin river, leak most in a random split. `KFold` does not use groups, so `None` is passed for it.
</details>

## 7. Which features did the forest use?

A random forest can say how much each feature helped. `forest.feature_importances_` gives one number per feature; together they add up to 1. A feature whose questions separate the classes well, in many trees, gets a high importance.

Read importances with one caution. **Correlated features share importance.** B08 and B8A, or the three red-edge bands, carry almost the same information. The forest uses one or the other more or less by chance, and each looks less important than the information really is. Remove one of them, and its partner takes over its share.

So importances describe **this model**, not how the land works. They answer "what did the forest lean on?", not "what causes land cover?".

In [ ]:
importance = pd.Series(forest.feature_importances_, index=FEATURES).sort_values()
print(importance.round(3).to_string())
print("total:", round(importance.sum(), 3))

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(importance.index, importance.values, color="seagreen")
ax.set_xlabel("Importance (share of the forest's splitting power)")
ax.set_title("Random forest feature importance, 11 November 2021")
plt.show()

print("correlation of B08 and B8A:", round(after_table["B08"].corr(after_table["B8A"]), 3))

**NDVI** is the most important feature (about 0.161), then **MNDWI** (about 0.113) and the blue band B02 (about 0.105). The two indices together carry more than a quarter of the importance: each condenses two bands into one number that separates vegetation, bare ground and water, which is why D2 built them.

The NIR bands B08 and B8A get only about 0.06 each, although NIR is what makes vegetation bright. Their correlation is about **0.90**: they are near-twins, and B08 also feeds into NDVI. Their information is spread over several features.

The bar chart sorts the features, so the most important one is at the top. Importances always add up to 1 (here `1.0`).

### Practice

**Q1 · one line:** Print the three most important features with `.tail(3)` on `importance`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
importance.tail(3)
```

B02 (about 0.105), MNDWI (about 0.113) and NDVI (about 0.161). `importance` is sorted from small to large, so the last three are the largest.
</details>

**Q2 · two lines:** Add up the importance of the four NIR and red-edge bands B06, B07, B08 and B8A with `.loc` and `.sum()`, and compare it with NDVI's importance.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
nir_like = importance.loc[["B06", "B07", "B08", "B8A"]].sum()
print(round(nir_like, 3), round(importance["NDVI"], 3))
```

About 0.231 together, more than NDVI's 0.161. Spread over four correlated bands, NIR information looks small band by band and large in total.
</details>

**Q3 · three lines:** Train a forest **without NDVI** (use `[f for f in FEATURES if f != "NDVI"]`, A3), and print its three most important features. Who takes over NDVI's share?

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
no_ndvi = [f for f in FEATURES if f != "NDVI"]
forest_no_ndvi = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1).fit(X[no_ndvi], y)
print(pd.Series(forest_no_ndvi.feature_importances_, index=no_ndvi).sort_values().tail(3).round(3))
```

MNDWI rises to the top (about 0.132), with the red band B04 (about 0.127) and B02 (about 0.116) next. Without NDVI the forest leans on the bands NDVI was made from and on the other index. The importance moved; the information did not disappear.
</details>

## 8. A second date: land-cover change, and a saved map

The forest has learned how 2021 features lead to classes. The 2020 image has the **same 12 features**, so `forest.predict` can classify it too. Comparing the two maps cell by cell gives a **change table**: rows = class in 2020, columns = class in 2021, each entry an area in km².

Be careful. The model learned from 2021 only. In 2020 the satellite (S2A instead of S2B), the date and the light were different (D3). A shaded slope that looks like forest in one image may look like grassland in the other. Every error of either map becomes a "change". Expect much **false change**, and trust most the changes that are large, connected and plausible, like D3's strip.

Finally, save the 2021 map as a GeoTIFF: `uint8`, nodata 0, one band, the image's grid. `dst.write_colormap(1, {...})` stores a colour for each code inside the file, so QGIS shows the official colours at once. Colours are written as (red, green, blue, opacity) from 0 to 255.

In [ ]:
valid_both = valid & before_table.notna().all(axis=1).to_numpy()
predicted_2020 = np.zeros(len(before_table), dtype="uint8")
predicted_2020[valid_both] = forest.predict(before_table.loc[valid_both, FEATURES])
rf_map_2020 = predicted_2020.reshape(n_rows, n_cols)

change_km2 = pd.crosstab(pd.Series(predicted_2020[valid_both]).map(NAMES), pd.Series(predicted[valid_both]).map(NAMES),
                         rownames=["2020"], colnames=["2021"]) * CELL_KM2
print(change_km2.round(2))
same = (predicted_2020[valid_both] == predicted[valid_both]).mean()
print(f"same class on both dates: {same * 100:.1f}% of cells")

In [ ]:
lost_green = np.isin(rf_map_2020, [10, 30, 40]) & np.isin(rf_map, [60, 80])    # green in 2020, bare or water in 2021
print("green -> bare/water:", lost_green.sum(), "cells =", round(lost_green.sum() * CELL_KM2, 2), "km²")
print("of these, in D3's flood strip:", (lost_green & strip).sum(), "cells")
print("share of the strip that became bare/water:", round((lost_green & strip).sum() / strip.sum() * 100, 1), "%")

fig, axes = plt.subplots(1, 3, figsize=(13, 8), sharey=True)
for ax, picture, title in zip(axes, [lut[rf_map_2020], lut[rf_map]], ["Random forest, 29 Oct 2020", "Random forest, 11 Nov 2021"]):
    ax.imshow(picture, extent=extent_km, interpolation="nearest")
    ax.set_title(title)
axes[2].imshow(get(after, "B08"), extent=extent_km, cmap="gray", vmin=0, vmax=0.5, alpha=0.5)
axes[2].imshow(np.where(lost_green, 1.0, np.nan), extent=extent_km, cmap=ListedColormap(["magenta"]), interpolation="nearest")
axes[2].contour(strip.astype(float), levels=[0.5], extent=extent_km, origin="upper", colors="black", linewidths=0.6)
axes[2].set_title("Green in 2020, bare/water in 2021 (magenta)\nblack outline: D3's strip")
for ax in axes:
    ax.axis([355, 362, 3076, 3092])
    ax.set_xlabel("Easting (km)")
axes[0].set_ylabel("Northing (km)")
legend_and_credits(fig)
plt.show()

In [ ]:
map_path = OUT / "d04_landcover_2021.tif"
out_profile = profile.copy()                     # WorldCover's profile: same grid, uint8, nodata 0
out_profile.update(count=1, dtype="uint8", nodata=0)
colormap = {c: tuple(round(v * 255) for v in to_rgb(COLOURS[c])) + (255,) for c in CODES}

with rasterio.open(map_path, "w", **out_profile) as dst:
    dst.write(rf_map, 1)
    dst.write_colormap(1, colormap)
    dst.set_band_description(1, "Random forest land cover, Sentinel-2 11 Nov 2021, trained on ESA WorldCover 2021")

with rasterio.open(map_path) as src:
    back = src.read(1)
    print(src.dtypes, src.nodata, src.crs, src.res, "| same transform:", src.transform == transform)
    print("colour of built-up (50):", src.colormap(1)[50])
print("values equal:", np.array_equal(back, rf_map), "| codes:", np.unique(back))

Read the change table along its diagonal first: those are cells with the same class on both dates. Only about **79.1%** of the cells keep their class, so about a fifth of the valley "changed". That is not believable for one year. The largest entries off the diagonal are about **18.8 km²** of grassland in 2020 that became tree cover in 2021 and about 6.1 km² the other way. Forest does not grow on grassland in a year: these are mostly **false changes** from different light, season and satellite (D3), and from the uncertain border between grassland and tree cover (step 5).

The flood shows up differently. About **4.45 km²** (11,126 cells) were green (tree cover, grassland or cropland) in 2020 and bare or water in 2021. Of these, 5,187 cells lie in D3's flood strip, where they cover about **67.1%** of the strip. In the right-hand map, the magenta cells fill the black outline of the strip along the river. Yet fewer than half of all magenta cells (5,187 of 11,126) lie inside the strip; the rest are scattered over the slopes as specks: false alarms again, like the specks of D3.

So a classification-change map agrees with D3 on the strip, adds a *name* to each change (from cropland to bare, from forest to water), and adds much noise. D3's simpler NDVI difference measured the same strip more cleanly.

The saved file is `uint8` with nodata 0 in EPSG:32645 with 20 m cells, on WorldCover's grid. Read back, its values equal `rf_map`, and it carries the colours: built-up (50) is `(250, 0, 0, 255)`, the official red. Open it in QGIS and the classes appear in WorldCover's colours.

*Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.*  
*Reference labels: © ESA WorldCover project 2021 / Contains modified Copernicus Sentinel data (2021) processed by ESA WorldCover consortium (CC BY 4.0).*

### Practice

**Q1 · one line:** How many km² did the forest call **water** in 2020, and in 2021? Use `(rf_map_2020 == 80).sum()` and the same for `rf_map`, times `CELL_KM2`, in one `print`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(round((rf_map_2020 == 80).sum() * CELL_KM2, 2), round((rf_map == 80).sum() * CELL_KM2, 2))
```

About 0.71 km² in 2020 and 2.87 km² in 2021, four times more. Part of that is real: D2 found that the water-like area (MNDWI above 0) grew about four times too, as the river spread over its new bed. Part is the forest's generosity with water: only about 12% of its 2021 water cells are water in WorldCover (step 5).
</details>

**Q2 · two lines:** How much of the **strip** was each class in 2020, according to the forest? Use `pd.Series(rf_map_2020[strip]).map(NAMES).value_counts()`, as km².

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
in_strip_2020 = pd.Series(rf_map_2020[strip]).map(NAMES).value_counts()
print((in_strip_2020 * CELL_KM2).round(2))
```

About 1.03 km² grassland, 0.95 km² cropland and 0.46 km² tree cover in 2020, then 0.37 km² built-up, 0.24 km² bare / sparse and 0.03 km² water. Some of the built-up cells are probably bright gravel, the confusion of step 5. Before the flood, the strip was green ground beside a narrow river: what D1 and D3 saw by eye.
</details>

**Q3 · three lines:** Open the saved map with rioxarray (`masked=True`), and print its shape and the area of each class in km² from `np.unique(..., return_counts=True)` on the non-`NaN` values.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
saved = rxr.open_rasterio(map_path, masked=True).squeeze("band", drop=True)
codes, n = np.unique(saved.values[~np.isnan(saved.values)], return_counts=True)
print(saved.shape, dict(zip(codes.astype(int), (n * CELL_KM2).round(2))))
```

(1006, 702), and six classes, from about 182.88 km² of tree cover to about 2.87 km² of water. `masked=True` turns the 145 nodata cells (0) into `NaN`, so they are not counted as a class. Reading the file back proves it holds what you meant to save.
</details>

## 9. When it breaks: features the model has never seen

You want to classify the 2020 image quickly, but you build its table by hand and forget MNDWI. Run the cell.

In [ ]:
quick_table = before_table[BANDS + ["NDVI"]]       # only 11 features: MNDWI forgotten
forest.predict(quick_table.loc[valid_both])

The last lines of the error say:

`ValueError: The feature names should match those that were passed during fit.`
`Feature names seen at fit time, yet now missing:`
`- MNDWI`

- **ValueError**: the value you passed, the table, cannot be used.
- **feature names … passed during fit**: when you fitted the forest with a DataFrame, it remembered the 12 column names, in order. They are in `forest.feature_names_in_`.
- **yet now missing: MNDWI**: it even names the missing column.

The model does not know what a band *is*. It knows only "column 1, column 2, …". If you swap two columns, every tree asks its questions about the wrong numbers. With column names, scikit-learn catches this: the same error appears, with *"Feature names must be in the same order as they were in fit"*, if all 12 columns are there in another order.

A plain NumPy array has no names. With 11 columns it fails with *"X has 11 features, but RandomForestClassifier is expecting 12 features as input"*. With 12 columns in the wrong order it would **not fail at all**: it gives a warning that *X does not have valid feature names*, then a wrong map. That is why this lesson keeps its features in DataFrames and always selects them with `[FEATURES]`.

**Try it:** fix the cell by selecting `before_table.loc[valid_both, FEATURES]`.

## 10. What this can and cannot tell you

**What it can tell you**

- How the 12 features of a 2021 Sentinel-2 image sort the Melamchi valley into six land-cover classes, in a map that reproduces WorldCover's broad pattern: forest on the slopes, fields and grassland lower down, bare ground and water along the river.
- Where the model agrees and disagrees with the reference labels, class by class: high recall for most classes, low precision for the rare ones, and bare / sparse as the weakest class.
- An honest score: about 0.65 in a spatial cross-validation on a balanced sample, a little below the random-split score of about 0.68.

**What it cannot tell you**

- **Whether the map is right on the ground.** Every score here compares one model with another model's map. WorldCover has its own errors, such as cropland and grassland labels on the 2021 flood sediment (step 2). A field survey, or careful interpretation of very detailed images, would be needed for a true accuracy.
- **Areas you can quote.** The balanced sample makes the forest call nearly four times too many cells built-up or water (review 2). Class areas from this map are biased; a careful study would correct them with the confusion matrix, or use a sample that matches the real class shares.
- **Change you can trust cell by cell.** Applied to 2020, the same model "changed" about a fifth of the valley, mostly false change from light, season, satellite and uncertain class borders. Only large, connected, plausible changes, such as the flood strip, stand out of that noise.
- **Anything about causes.** A class says what a cell looks like from space at 20 m, not why it looks that way or what lies beneath.

*Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.*  
*Reference labels: © ESA WorldCover project 2021 / Contains modified Copernicus Sentinel data (2021) processed by ESA WorldCover consortium (CC BY 4.0).*

## Mixed review

These exercises combine everything in this lesson. Each one needs 5 to 10 lines. Plan in words first, then write the code.

### Review 1 · Do the indices help?

Train a forest on the **10 bands only** (`BANDS`) and compare its **spatial** cross-validation accuracy (`GroupKFold(5)` with `groups=blocks`) with the 12-feature model's `spatial_scores`. Print both means, rounded to 3 decimals.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
bands_only = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1)
scores_bands = cross_val_score(bands_only, X[BANDS], y, cv=GroupKFold(5), groups=blocks)
print("10 bands   :", round(scores_bands.mean(), 3))
print("12 features:", round(spatial_scores.mean(), 3))
```

About 0.654 with the 10 bands and 0.650 with all 12 features: no real difference, well within the spread between folds. NDVI and MNDWI are made from bands the forest already has, and a forest can combine bands by itself with many yes/no questions. The indices still made the model easier to read (step 7), which is worth something, but they did not add information.
</details>

### Review 2 · Class areas: model against reference

Build a table with one row per class: its name, the area in km² in WorldCover, the area in the forest's 2021 map, and the ratio forest / WorldCover. Round to 2 decimals. Which classes does the forest over- or under-estimate?

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
rows = []
for c in CODES:
    rows.append({"class": NAMES[c], "worldcover_km2": (worldcover == c).sum() * CELL_KM2,
                 "forest_km2": (rf_map == c).sum() * CELL_KM2})
compare = pd.DataFrame(rows)
compare["ratio"] = compare["forest_km2"] / compare["worldcover_km2"]
print(compare.round(2))
```

Built-up and water are overestimated about 3.8 times, cropland about 2.1 times and grassland about 1.4 times; bare / sparse comes out about right in total (1.01), although step 5 showed that most of its cells are not in the same places. Tree cover is about 17% too small. These are the false alarms of step 5, added up: a balanced training sample shifts the areas towards the rare classes.
</details>

### Review 3 · Where do the two maps disagree?

Make a mask of valid cells where `rf_map` differs from `worldcover`. Print the share of valid cells that disagree, and, for each WorldCover class, the percentage of its cells that the forest labels differently. Then map the disagreement in black on a pale NIR background.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
disagree = (rf_map != worldcover) & valid.reshape(n_rows, n_cols)
print("disagree:", round(disagree.sum() / valid.sum() * 100, 1), "% of valid cells")
for c in CODES:
    print(f"{NAMES[c]:15} {disagree[worldcover == c].mean() * 100:5.1f} %")

fig, ax = plt.subplots(figsize=(7, 9))
ax.imshow(get(after, "B08"), extent=extent_km, cmap="gray", vmin=0, vmax=0.5, alpha=0.4)
ax.imshow(np.where(disagree, 1.0, np.nan), extent=extent_km, cmap=ListedColormap(["black"]), interpolation="nearest")
ax.set_title("Random forest and WorldCover disagree (black)")
fig.text(0.01, 0.02, S2_CREDIT + "\n" + WC_CREDIT, fontsize=7)
plt.show()
```

About 21.8% of the valid cells disagree. By WorldCover class, the forest relabels about 18.5% of tree cover, 24% to 33% of built-up, cropland and grassland, only about 8% of water, and about 58% of bare / sparse. On the map the disagreement is not random noise: it follows the valley sides and field areas, where grassland, cropland and tree cover meet, and the edges of the river bed. Disagreement maps show where neither map should be trusted without a closer look.
</details>

### Review 4 · North against south

The harshest spatial test: train on the training cells in the **northern half** of the image (`cell_row < n_rows // 2`) and test on the southern half, then the other way round. Print both accuracies, rounded to 3 decimals, and a `pd.crosstab` of half (north/south) against class name, to see why.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
north = cell_row < n_rows // 2
for name, part in [("north -> south", north), ("south -> north", ~north)]:
    half_forest = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1)
    half_forest.fit(X[part], y[part])
    print(name, round(accuracy_score(y[~part], half_forest.predict(X[~part])), 3))

half = np.where(north, "north", "south")
print(pd.crosstab(half, y.map(NAMES).values))
```

About 0.514 from north to south and 0.615 from south to north, both far below the cross-validation scores. The table shows why: the south holds 883 of the 1,000 water cells, 843 of the cropland and 689 of the built-up cells, so a model trained in the north has seen few of them. A model is only as good as the range of places it learned from; a landslide model meets exactly the same problem.
</details>

### Review 5 · How many trees are enough?

Compare forests of 10, 50 and 100 trees with the spatial cross-validation of step 6. For each, print the number of trees, the mean accuracy (3 decimals) and the time it took in seconds (1 decimal), measured with `time.perf_counter()` from the `time` module before and after `cross_val_score`.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
import time

for n_trees in [10, 50, 100]:
    start = time.perf_counter()
    model_n = RandomForestClassifier(n_estimators=n_trees, random_state=0, n_jobs=-1)
    scores_n = cross_val_score(model_n, X, y, cv=GroupKFold(5), groups=blocks)
    seconds = time.perf_counter() - start
    print(n_trees, "trees:", round(scores_n.mean(), 3), "accuracy,", round(seconds, 1), "s")
```

About 0.632 with 10 trees, 0.647 with 50 and 0.650 with 100. More trees help a little, and the gain shrinks as the forest grows. The time grows roughly with the number of trees; your times depend on your computer. A modest forest is the sensible start; add trees only when the score still improves.
</details>

## Recap

- **Classification** gives every cell a class from its **features**. To a model, an image is a **table**: `reshape(bands, -1).T` gives one row per cell, and `.reshape(rows, cols)` turns any column or prediction back into a map. Drop rows with missing features.
- **scikit-learn** models share one pattern: `fit` learns, `predict` applies. A **pipeline** chains steps such as `StandardScaler` and `KMeans`. **k-means** (unsupervised) finds clusters, which you must name, for example with `pd.crosstab` against reference labels; clusters are not classes.
- A **random forest** (supervised) votes with many decision trees. Train it on a **stratified** sample (here 1,000 cells per class), because tree cover fills 78% of the valley; expect more false alarms for rare classes in return.
- Test on cells the model has not seen. Report the **confusion matrix**, **precision** and **recall**, not only overall accuracy: a "tree cover everywhere" map agrees with WorldCover on about 78% of this image, as often as the forest map does. Neighbouring cells leak; a **spatial** cross-validation with blocks (`GroupKFold`) is the more honest score.
- **Feature importance** shows what a model leaned on; correlated features share it. Applying a model to another date gives a change table full of false change; save class maps as `uint8` GeoTIFFs with nodata 0 and a colour map, and call WorldCover what it is: **reference labels**, not truth.

### Words you learned

| Word | Meaning |
| --- | --- |
| classification | Giving every cell a class, such as forest or water, from its features |
| feature | A number that describes a cell and is used to classify it, such as a band or an index |
| label / reference label | The known class of a cell, used to train or test a model; here from WorldCover |
| unsupervised / supervised | Finds groups without labels / learns from labelled examples |
| scikit-learn (`sklearn`) | The standard Python library for machine learning |
| model | A recipe with numbers that it learns from data, then applies to new data |
| `fit` / `predict` | Learn from a table / apply what was learned to a table |
| `X` / `y` | By custom, the table of features / the list of labels |
| transpose (`.T`) | Swapping the rows and columns of an array |
| k-means / cluster | A method that sorts cells into k groups around k centres / one such group |
| `StandardScaler` | Rescales each feature to a mean of 0 and a standard deviation of 1 |
| pipeline | A chain of steps (such as scaling, then a model) that run together as one model |
| decision tree | A chain of yes/no questions about features that ends in a class |
| random forest | Many decision trees, each grown on random parts of the data, that vote |
| stratified sample | A sample with a chosen number of cells from each class |
| class imbalance | One class far more common than the others |
| hold-out / test set | Labelled cells kept apart from training and used only to test |
| overall accuracy | The share of test cells whose predicted class is the reference class |
| confusion matrix | A table of reference classes against predicted classes |
| precision / recall | Of the cells called class A, the share that are A / of the cells that are A, the share found |
| leakage | Test cells that are very similar to nearby training cells, which makes a score too optimistic |
| cross-validation / fold | Repeating train-and-test so every cell is tested once / one part of the data |
| spatial cross-validation | Cross-validation that keeps whole blocks of the map together |
| feature importance | How much each feature helped a random forest to separate the classes |
| colour map (in a GeoTIFF) | A table stored in the file that gives each class code a colour |

## Next

**This is the end of Part D.** You can now read satellite images, build indices, map change and classify land cover. From here, [PyGILE-Plus](https://github.com/Geoinformatics-Lab/PyGILE-Plus) builds on this course with landslide susceptibility modelling and deep learning, and uses the spatial testing of this lesson.

<!-- pygile-nav -->
**PyGILE Course** · lesson 36 of 36 · [Course home](../README.md)  
← Previous: [D3 · Change detection: the 2021 Melamchi flood](D03_Change_detection_Melamchi_flood.ipynb) · You have reached the end of the course